# Making UI With Gradio

## Importing the libraries

In [1]:
import os
import ollama
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

## Loading OpenAI API Key

In [2]:
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found!")
else:
    print("API key found!")

API key found!


## Creating user interface

In [3]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [4]:
interface = gr.Interface(
    fn = shout,
    inputs = "textbox",
    outputs = "textbox",
    flagging_mode = "never"
)

In [5]:
interface.launch(
    inbrowser = True,
    share = True,
    auth = ("admin","admin123")
)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Shout has been called with input Hey


In [6]:
interface.close()

Closing server running on port: 7860


## Using Javascipt in interface

In [7]:
dark_mode_js = """
function darkMode(){
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark'){
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

In [ ]:
interface = gr.Interface(
    fn = shout,
    inputs = "textbox",
    outputs = "textbox",
    flagging_mode = "never",
    js = dark_mode_js
)

In [9]:
interface.launch(
    inbrowser = True
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Hey


In [10]:
interface.close()

Closing server running on port: 7860


## Adding input labels

In [11]:
message_input = gr.Textbox(
    label = "Enter you message:",
    info = "Enter a message to shout",
    lines = 6
)

In [12]:
message_output = gr.Textbox(
    label = "Response:",
    info = "Output message",
    lines = 6
)

In [13]:
interface = gr.Interface(
    fn = shout,
    title = "Shout",
    inputs = [message_input],
    outputs = [message_output],
    examples =["Hey", "Howdy"],
    flagging_mode = "never"
)

In [14]:
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
interface.close()

Closing server running on port: 7860


## Setting up GPT and Ollama

In [16]:
openai = OpenAI()

In [17]:
system_message = "You are a helpful assistant"

In [18]:
def call_gpt(prompt):
    messages = [
        { "role": "system", "content": system_message},
        { "role": "user", "content": prompt}
    ]

    stream = openai.responses.create(
        model = "gpt-5.4-mini",
        input = messages,
        stream = True
    )

    response = ""
    for event in stream:
        if event.type == "response.output_text.delta":
            response += event.delta
            yield response

In [19]:
def call_ollama(prompt):
    messages = [
        { "role": "system", "content": system_message},
        { "role": "user", "content": prompt}
    ]

    stream = ollama.chat(
        model = "qwen3:0.6b",
        messages = messages,
        stream = True
    )

    response = ""
    for chunk in stream:
        if chunk["message"]["content"]:
            response += chunk["message"]["content"]
            yield response

In [20]:
def call_model (prompt, model):
    
    if model == 'GPT':
        yield from call_gpt(prompt)
    elif model == "Ollama":
        yield from call_ollama(prompt)
    else:
        raise ValueError("Incorrect Model!")

## Creating interface to chat with model

In [21]:
prompt_input = gr.Textbox(
    label = "Ask anything:",
    info = "Enter your prompt for the LLM",
    lines = 6
)

In [22]:
model_selector = gr.Dropdown(
    ["GPT", "Ollama"],
    label = "Select a model:",
    value = "GPT"
)

In [23]:
prompt_output = gr.Markdown(
    label = "Response:",
    height = 350
)

In [24]:
interface = gr.Interface(
    fn = call_model,
    title = "Chat with LLM",
    inputs = [prompt_input, model_selector],
    outputs = [prompt_output],
    examples = [
        ["Explain the Transformer Architecture", "GPT"],
        ["What is an MCP Server?", "Ollama"]
    ],
    flagging_mode = "never"
)

In [25]:
interface.launch(
    inbrowser = True
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [26]:
interface.close()

Closing server running on port: 7860


## Creating chat interface with features

In [39]:
def call_model_chat (prompt, history, model):
    
    if model == 'GPT':
        yield from call_gpt(prompt)
    elif model == "Ollama":
        yield from call_ollama(prompt)
    else:
        raise ValueError("Incorrect Model!")

In [ ]:
model_selector_chat = gr.Dropdown(
    ["GPT", "Ollama"],
    label = "Select a model:",
    value = "GPT"
)

In [ ]:
interface = gr.ChatInterface(
    fn = call_model_chat,
    title = "Chat with LLM",
    additional_inputs = [model_selector_chat],
    flagging_mode = "never"
)

In [42]:
interface.launch(
    inbrowser = True
)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [43]:
interface.close()

Closing server running on port: 7862
